Pada tahap ini dilakukan pengujian model terbaik menggunakan data baru yang belum pernah dilihat sebelumnya oleh model. Model yang digunakan pada tahap inference adalah **Improvement Experiment 3**

Tujuan inference adalah untuk melihat kemampuan model dalam melakukan prediksi sentimen terhadap komentar baru secara langsung.

# 1. Import Libraries

In [1]:
import pandas as pd
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import warnings
warnings.filterwarnings('ignore')

# 2. Load Data

In [2]:
# Load best model
model_inference = load_model(
    'best_sentiment_model.keras'
)

# Load tokenizer
with open('tokenizer.pkl', 'rb') as file:
    tokenizer = pickle.load(file)

# 3. Input Data Baru

In [3]:
new_text = [
    "Keren banget penampilannya, salut sama perjuangannya",
    "Juri nya tidak adil dan sangat mengecewakan",
    "Semangat terus buat tim nya",
    "Keputusan juri sangat buruk dan merugikan peserta",
    "Pembawa acaranya sangat profesional",
]

# 4. Preprocessing Data Baru

In [4]:
# Case folding sederhana
new_text_clean = [

    text.lower()

    for text in new_text
]

new_text_clean

['keren banget penampilannya, salut sama perjuangannya',
 'juri nya tidak adil dan sangat mengecewakan',
 'semangat terus buat tim nya',
 'keputusan juri sangat buruk dan merugikan peserta',
 'pembawa acaranya sangat profesional']

# 5. Tokenisasi dan Padding

In [5]:
MAX_LENGTH = 100

# Mengubah teks menjadi sequence numerik
new_sequences = tokenizer.texts_to_sequences(
    new_text_clean
)

# Padding sequence
new_padded = pad_sequences(

    new_sequences,

    maxlen=MAX_LENGTH,

    padding='post',

    truncating='post'
)

new_padded

array([[509,   1,   1,   1,   1,   1,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0],
       [  2,   1,   3,  42,   1,  21, 753,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   

# 6. Prediksi Sentimen

In [6]:
# Prediksi probabilitas
prediction_prob = model_inference.predict(
    new_padded
)

# Konversi probabilitas menjadi label
prediction_label = (
    prediction_prob > 0.5
).astype(int)

prediction_label

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 786ms/step


array([[1],
       [0],
       [1],
       [0],
       [1]])

# 7. Konversi Label Sentimen

In [7]:
# Konversi label ke sentimen
sentiment_result = []

for label in prediction_label:
    if label == 1:
        sentiment_result.append('Positif')
    else:
        sentiment_result.append('Negatif')

# Membuat dataframe hasil inference
inference_result = pd.DataFrame({
    'Text': new_text,
    'Prediction Probability': prediction_prob.flatten(),
    'Predicted Sentiment': sentiment_result
})

print("="*50)
print("Inference Result")
print("="*50)

inference_result

Inference Result


,Text,Prediction Probability,Predicted Sentiment
0,"Keren banget penampilannya, salut sama perjuan...",0.836441,Positif
1,Juri nya tidak adil dan sangat mengecewakan,0.009564,Negatif
2,Semangat terus buat tim nya,0.645732,Positif
3,Keputusan juri sangat buruk dan merugikan peserta,0.006058,Negatif
4,Pembawa acaranya sangat profesional,0.600611,Positif


Insight:

- Berdasarkan hasil inference, model mampu melakukan klasifikasi sentimen komentar YouTube dengan cukup baik. Komentar yang mengandung kata positif seperti “keren”, “salut”, dan “semangat” berhasil diprediksi sebagai sentimen positif dengan nilai probability yang cukup tinggi. Sebaliknya, komentar yang mengandung sentimen negatif seperti “tidak adil”, “mengecewakan”, “buruk”, dan “merugikan” berhasil diprediksi sebagai sentimen negatif dengan probability yang sangat rendah mendekati 0.

- Hasil ini menunjukkan bahwa model mampu memahami pola kata dan konteks komentar yang mirip dengan data training sehingga performa inference menjadi lebih optimal. Selain itu, nilai prediction probability juga menunjukkan tingkat keyakinan model terhadap hasil prediksi yang diberikan. Secara keseluruhan, model inference berhasil menunjukkan kemampuan klasifikasi sentimen yang cukup baik pada komentar media sosial atau YouTube.